In [1]:
import numpy as np
import pandas as pd
import data_management as dm
import feature_builders as fb
import model_wrappers as wp

In [2]:
TICKERS = ["AAPL", "JPM", "XOM", "KO"]

WEIGHTS = {
    "AAPL": 0.25,
    "JPM": 0.25,
    "XOM": 0.25,
    "KO": 0.25,
}

df = pd.read_csv(
    "./data_general/4_ticker_data.csv",
    parse_dates=["Date"],
    index_col="Date",
)


tickers_data = df[TICKERS].pct_change().dropna()



In [3]:
calendar_last_180_days = dm.DailyGroundTruthCalendar(
    df, TICKERS, WEIGHTS,
    train_start="2024-06-03",   # Pazartesi, ~6 ay geriye (180 takvim günü civarı)
    train_end="2024-11-29",     # Cuma, Kasım'ın son iş günü
    forecast_start="2024-12-02",  # Pazartesi
    forecast_end="2024-12-31",
)

[WARNING] forecast_end=2024-12-31 falls on Tuesday, expected Friday (using it as-is).
[WARNING] [train] 4 weekday days had no data and were skipped (likely market holidays): 2024-06-19, 2024-07-04, 2024-09-02, 2024-11-28
[INFO] train set: 126 bars, 4 weekday days skipped
[WARNING] [forecast] 1 weekday day had no data and were skipped (likely market holidays): 2024-12-25
[INFO] forecast set: 21 bars, 1 weekday day skipped


In [4]:
training_feature, forecast_feature = fb.build_long_features(calendar_last_180_days)

[INFO] long train: 504 rows -> 424 after dropna (80 dropped, 15.9%)
[INFO] long forecast: 84 rows -> 84 after dropna (0 dropped, 0.0%)


In [5]:
training_feature.X_in_sample

,r,abs_r,real_vol_5,mean_abs_5,skew_5,real_vol_10,mean_abs_10,skew_10,real_vol_21,mean_abs_21,skew_21,ewma_vol,market_ewma_vol
0,218.579498,218.579498,3.642393,213.347968,0.521779,4.077315,210.736163,0.679073,8.743396,205.837575,-0.586058,7.204697,2.670447
1,200.203049,200.203049,4.433392,194.257260,0.333618,3.901906,191.935316,1.366858,3.836601,190.143664,0.859171,4.502730,2.670447
2,60.051090,60.051090,0.370035,60.486615,-0.122962,0.616779,60.234619,-0.609381,0.553848,60.046875,-0.016465,0.561156,2.670447
3,107.650177,107.650177,0.376392,108.153636,-0.603262,1.899138,106.902525,-1.216077,1.980171,106.156942,-0.659424,2.036931,2.670447
4,219.849670,219.849670,4.437457,214.995227,-0.338612,4.987108,211.456592,0.664978,8.697595,207.137993,-0.666476,7.584268,3.014613
...,...,...,...,...,...,...,...,...,...,...,...,...,...
419,112.743134,112.743134,1.942228,114.855023,-0.076872,1.397775,114.836813,-0.088278,1.927361,114.040846,-0.934173,1.830498,3.200961
420,236.040573,236.040573,2.787020,232.740607,-0.645631,3.909638,229.633585,0.336691,4.361706,226.412816,0.773024,4.571039,3.284139
421,242.205612,242.205612,0.640712,242.151291,-1.599066,3.268829,239.304608,-0.484167,9.701890,232.580416,-1.049021,10.954390,3.284139
422,61.818546,61.818546,0.251964,61.630103,-1.757656,1.058161,60.777399,-0.627254,1.020455,61.047552,-0.264206,1.992055,3.284139


In [6]:
training_feature.y_in_sample

,target
0,48333.877579
1,40027.537697
2,3626.721001
3,11706.592156
4,50446.471912
...,...
419,12775.913426
420,NaN
421,NaN
422,NaN


In [7]:
forecast_feature.X_forecast_sample

,r,abs_r,real_vol_5,mean_abs_5,skew_5,real_vol_10,mean_abs_10,skew_10,real_vol_21,mean_abs_21,skew_21,ewma_vol,market_ewma_vol
0,238.288284,238.288284,2.558599,234.674039,0.474665,4.178281,231.084656,0.532020,5.042170,227.072463,0.789330,5.189775,3.281647
1,238.840027,238.840027,1.615844,241.705133,-2.136371,3.238752,239.395781,-0.593128,8.929120,233.704124,-1.341593,10.561854,3.281647
2,61.403721,61.403721,0.167272,61.670844,-1.205606,0.892800,61.006493,-1.081878,0.966766,60.993879,-0.323267,1.900891,3.281647
3,112.925186,112.925186,0.912049,113.339136,2.137962,1.572510,114.447781,0.420099,1.682290,114.234443,-1.127107,1.746971,3.281647
4,241.331665,241.331665,3.246815,236.619409,0.732636,4.971385,232.539703,0.489819,5.750778,228.018957,0.875065,6.036787,3.282624
...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,101.340408,101.340408,0.291082,101.844429,-1.906392,0.908596,102.117520,1.258632,4.175742,105.949405,0.398968,4.892110,2.070288
80,249.059464,249.059464,3.705615,253.700110,-0.277241,3.562238,252.272906,-0.027292,5.386660,247.968279,0.092895,8.933913,1.976952
81,232.496826,232.496826,1.588291,233.874091,0.092093,3.861578,231.161256,-1.012458,4.018785,233.789109,-0.957485,5.258429,1.976952
82,60.062771,60.062771,0.296168,60.226775,0.039753,0.365595,60.369552,0.981999,0.471254,60.559370,0.792446,0.939252,1.976952


In [8]:
training_feature2, forecast_feature2 = fb.build_wide_features(calendar_last_180_days)


[INFO] wide train: 126 bars -> 106 after dropna (20 dropped, 15.9%)
[INFO] wide forecast: 21 bars -> 21 after dropna (0 dropped, 0.0%)


In [9]:
training_feature2.X_in_sample

,AAPL,JPM,XOM,KO,portfolio_return,AAPL_abs_r,JPM_abs_r,XOM_abs_r,KO_abs_r,portfolio_return_abs_r,...,AAPL_skew_21,JPM_skew_21,XOM_skew_21,KO_skew_21,portfolio_return_skew_21,AAPL_ewma_vol,JPM_ewma_vol,XOM_ewma_vol,KO_ewma_vol,portfolio_return_ewma_vol
Date,,,,,,,,,,,,,,,,,,,,,
2024-07-02,218.579498,200.203049,107.650177,60.051090,146.620954,218.579498,200.203049,107.650177,60.051090,146.620954,...,-0.586058,0.859171,-0.659424,-0.016465,0.785990,7.204697,4.502730,2.036931,0.561156,2.670447
2024-07-03,219.849670,200.068832,108.197006,60.222263,147.084443,219.849670,200.068832,108.197006,60.222263,147.084443,...,-0.666476,0.975826,-0.646931,-0.158497,0.763428,7.584268,5.059952,1.990733,0.531293,3.014613
2024-07-05,224.602921,197.417816,106.886490,60.631157,147.384596,224.602921,197.417816,106.886490,60.631157,147.384596,...,-0.547971,0.788704,-0.690591,-0.185396,0.693611,8.447132,5.052021,1.885091,0.524112,3.252186
2024-07-08,226.071579,197.784149,105.764557,59.870415,147.372675,226.071579,197.784149,105.764557,59.870415,147.372675,...,-0.460996,0.569838,-0.622411,-0.063877,0.580761,9.146326,5.029920,1.813894,0.507552,3.379429
2024-07-09,226.924957,200.155579,104.595467,59.613667,147.822417,226.924957,200.155579,104.595467,59.613667,147.822417,...,-0.315479,0.408739,-0.433579,0.136504,0.507316,9.645001,5.216825,1.836555,0.511442,3.488527
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-11-22,228.621124,241.070816,116.700531,61.200016,161.898122,228.621124,241.070816,116.700531,61.200016,161.898122,...,0.237012,-0.304224,-0.759255,0.306640,-0.511463,3.173017,10.923054,1.954060,2.408906,2.800999
2024-11-25,231.604813,242.758469,114.956596,61.640438,162.740079,231.604813,242.758469,114.956596,61.640438,162.740079,...,0.300158,-0.441958,-0.848931,0.209299,-0.413935,3.403982,11.134655,1.865756,2.296804,3.015600
2024-11-26,233.782913,242.448059,113.040176,61.803204,162.768588,233.782913,242.448059,113.040176,61.803204,162.768588,...,0.484917,-0.612591,-0.852333,-0.035192,-0.388669,3.844139,11.175178,1.841112,2.190523,3.147958


In [10]:
training_feature2.y_in_sample

,target
Date,
2024-07-02,21633.833400
2024-07-03,21722.219100
2024-07-05,21718.705320
2024-07-08,21851.467044
2024-07-09,22258.971787
...,...
2024-11-22,26484.333289
2024-11-25,26493.613261
2024-11-26,26435.390306
